# RSNA Knee — student-prep (Fase B langkah 2a)
# Cache uint8 256 (FOV 128 mm, max 28 slice) + label konsensus 4 tabel ⊕ teacher r384 (50/50).
# Output: cache256_train/*.npy, cache256_train_meta.csv, labels_student_v0.csv → dataset prep alver.


In [ ]:

import os, sys, glob, json, time, shutil, subprocess
from multiprocessing import Pool
import numpy as np, pandas as pd

t0 = time.time()
def say(*a): print(f"[{time.time()-t0:7.0f}s]", *a, flush=True)

# ---- cari mount ----
def find(pats, base="/kaggle/input"):
    for p in pats:
        for f in glob.glob(os.path.join(base, "**", p), recursive=True):
            return f
    for p in pats:
        for f in glob.glob(os.path.join(base, p), recursive=True):
            return f
    return None

COMP = None
for c in ["/kaggle/input/competitions/rsna-knee-abnormality-detection",
          "/kaggle/input/rsna-knee-abnormality-detection"] + sorted(glob.glob("/kaggle/input/*")):
    if os.path.exists(os.path.join(c, "train.csv")) and os.path.exists(os.path.join(c, "train_series.csv")):
        COMP = c; break
    for sub in sorted(glob.glob(os.path.join(c, "*"))):
        if os.path.exists(os.path.join(sub, "train.csv")) and os.path.exists(os.path.join(sub, "train_series.csv")):
            COMP = sub; break
    if COMP: break
assert COMP, "competition mount tidak ketemu; listing: " + str(sorted(os.listdir("/kaggle/input"))[:30])
say("COMP =", COMP)
TRAIN = pd.read_csv(f"{COMP}/train.csv")
SERIES = pd.read_csv(f"{COMP}/train_series.csv")
say("train", len(TRAIN), "series rows", len(SERIES))

EXT = {}
EXT["steven"] = find(["llm_labels_v4_blend.csv"])
EXT["pilkwang"] = find(["report_labels_v2.csv"])
EXT["lixin"] = find(["labels_llm_gpt56sol.csv"])
EXT["qwen"] = find(["qwen_knee_weak_labels.csv"])
TEACHER = find(["teacher_r384_train_preds.csv"])
assert all(EXT.values()), (EXT, sorted(os.listdir("/kaggle/input")))
say("labels ok:", EXT); say("teacher:", TEACHER)

# pylibjpeg utk decode cepat (wheel dari dataset goodpjw)
whl_dir = os.path.dirname(find(["pylibjpeg-2.1.0-py3-none-any.whl"]) or "")
if whl_dir:
    for w in sorted(glob.glob(os.path.join(whl_dir, "*.whl"))):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", w], check=False)
say("pylibjpeg:", subprocess.run([sys.executable, "-m", "pip", "show", "pylibjpeg"], capture_output=True).returncode == 0)


In [ ]:

import cv2, pydicom, numpy as np, pandas as pd, os
from multiprocessing import Pool

DATA = COMP
WORK = "/tmp"          # ephemeral disk lebih besar dari /kaggle/working
MM_PER_PX = 0.5
SIZE = 256             # FOV 128 mm
MAX_SLICES = 28

CANON = {0: (+1, (1, +1), (2, -1)), 1: (+1, (0, +1), (2, -1)), 2: (-1, (0, +1), (1, +1))}
PLANES = ["Sagittal", "Coronal", "Axial"]

def _read(path):
    ds = pydicom.dcmread(path)
    arr = ds.pixel_array.astype(np.float32)
    if ds.get("PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    slope, inter = ds.get("RescaleSlope"), ds.get("RescaleIntercept")
    if slope is not None and inter is not None:
        arr = arr * float(slope) + float(inter)
    iop = [float(v) for v in ds.ImageOrientationPatient] if "ImageOrientationPatient" in ds else None
    ipp = [float(v) for v in ds.ImagePositionPatient] if "ImagePositionPatient" in ds else None
    ps = [float(v) for v in ds.PixelSpacing] if "PixelSpacing" in ds else None
    inst = float(ds.get("InstanceNumber") or 0)
    frames = list(arr) if arr.ndim == 3 else [arr]
    return [(f, iop, ipp, ps, inst + i * 1e-3) for i, f in enumerate(frames)]

def _canon_slice(img, iop, ps, plane):
    ps = ps or [MM_PER_PX * img.shape[0] / SIZE] * 2
    if iop is None:
        return img, ps
    r, c = np.array(iop[:3]), np.array(iop[3:])
    _, (cax, csgn), (rax, rsgn) = CANON[plane]
    if abs(r[cax]) < abs(r[rax]):
        img, r, c, ps = img.T, c, r, ps[::-1]
    if r[cax] * csgn < 0:
        img = img[:, ::-1]
    if c[rax] * rsgn < 0:
        img = img[::-1]
    return img, ps

def _resample(img, ps):
    out = np.zeros((SIZE, SIZE), np.float32)
    h, w = img.shape
    ch = min(h, int(round(SIZE * MM_PER_PX / ps[0])))
    cw = min(w, int(round(SIZE * MM_PER_PX / ps[1])))
    y0, x0 = (h - ch) // 2, (w - cw) // 2
    crop = np.ascontiguousarray(img[y0:y0 + ch, x0:x0 + cw])
    nh = max(1, min(SIZE, int(round(ch * ps[0] / MM_PER_PX))))
    nw = max(1, min(SIZE, int(round(cw * ps[1] / MM_PER_PX))))
    interp = cv2.INTER_AREA if nh < ch else cv2.INTER_LINEAR
    res = cv2.resize(crop, (nw, nh), interpolation=interp)
    oy, ox = (SIZE - nh) // 2, (SIZE - nw) // 2
    out[oy:oy + nh, ox:ox + nw] = res
    return out

def load_series(sdir, plane_name=None):
    slices, bad = [], 0
    for fn in os.listdir(sdir):
        try:
            slices.extend(_read(os.path.join(sdir, fn)))
        except Exception:
            bad += 1
    info = {"n_raw": len(slices), "n_bad": bad}
    if not slices:
        return np.zeros((1, SIZE, SIZE), np.uint8), info
    iop = next((s[1] for s in slices if s[1] is not None), None)
    if iop is not None:
        normal = np.cross(iop[:3], iop[3:]); plane = int(np.abs(normal).argmax())
    else:
        normal, plane = None, PLANES.index(plane_name) if plane_name in PLANES else 0
    info["plane"] = PLANES[plane]
    if normal is not None and all(s[2] is not None for s in slices):
        sgn = CANON[plane][0] * np.sign(normal[plane])   # urut by IPP (bukan filename)
        slices.sort(key=lambda s: float(np.dot(s[2], normal)) * sgn)
    else:
        slices.sort(key=lambda s: s[4])
    if len(slices) > MAX_SLICES:
        keep = np.linspace(0, len(slices) - 1, MAX_SLICES).round().astype(int)
        slices = [slices[i] for i in keep]
    vol = np.stack([_resample(*_canon_slice(s[0], s[1], s[3], plane)) for s in slices])
    lo, hi = np.percentile(vol[:, ::4, ::4], [1, 99.5])
    vol = np.clip((vol - lo) / max(hi - lo, 1e-6), 0, 1)
    info["n"] = len(vol)
    return (vol * 255).round().astype(np.uint8), info

def _job(args):
    study, series, plane, out_dir = args
    out = f"{out_dir}/{series}.npy"
    info = {"study": study, "series": series}
    if os.path.exists(out):
        return None
    try:
        vol, extra = load_series(f"{DATA}/train_series/{study}/{series}")
        np.save(out + ".tmp.npy", vol); os.replace(out + ".tmp.npy", out)
        info.update(extra)
    except Exception as e:
        info["error"] = f"{type(e).__name__}: {e}"[:200]
    return info

OUT = f"{WORK}/cache256_train"; os.makedirs(OUT, exist_ok=True)
tasks = [(a, b, p, OUT) for a, b, p in zip(SERIES.StudyInstanceUID, SERIES.SeriesInstanceUID, SERIES.Anatomical_Plane)]
rows = []
with Pool(4) as p:
    for i, r in enumerate(p.imap_unordered(_job, tasks, chunksize=8)):
        if r: rows.append(r)
        if i % 1000 == 0:
            say(f"cache {i}/{len(tasks)} files={len(os.listdir(OUT))} disk={shutil.disk_usage('/tmp').free/2**30:.1f}GB")
            if shutil.disk_usage("/tmp").free < 3 * 2**30:
                say("DISK HAMPIR HENTI — pakai cache parsial"); break
df = pd.DataFrame(rows)
if "error" in df: say("errors:", int(df["error"].notna().sum()))
meta = df.dropna(subset=["n"]).copy()
meta.to_csv("/kaggle/working/cache256_train_meta.csv", index=False)
say("meta rows", len(meta), "files", len(os.listdir(OUT)))


In [ ]:

import re, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
ids = TRAIN.StudyInstanceUID
gold = TRAIN.set_index("StudyInstanceUID")[LABELS]
is_gold = gold.notna().all(axis=1)

src = {}
src["steven"] = pd.read_csv(EXT["steven"])
src["pilkwang"] = pd.read_csv(EXT["pilkwang"])
src["lixin"] = pd.read_csv(EXT["lixin"])
q = pd.read_csv(EXT["qwen"])
src["qwen"] = q[["StudyInstanceUID"] + [f"{c}__label" for c in LABELS]].set_axis(["StudyInstanceUID"] + LABELS, axis=1)
src = {k: v.set_index("StudyInstanceUID")[LABELS].astype(float).reindex(ids) for k, v in src.items()}
num = sum(src[k].fillna(0) for k in src); den = sum(src[k].notna() for k in src)
cons = (num / den).clip(0.01, 0.99)

teacher = pd.read_csv(TEACHER).set_index("StudyInstanceUID")[LABELS].astype(float).reindex(ids)
alpha = 0.50
soft = ((1 - alpha) * cons.fillna(0.5) + alpha * teacher.fillna(0.5)).clip(0.01, 0.99)

print("AUC emas-58 (sumber vs label resmi):")
tab = {}
for k, v in {**src, "consensus": cons, "teacher": teacher, "blend50": soft}.items():
    tab[k] = [roc_auc_score(gold.loc[is_gold, c], v.loc[is_gold, c].fillna(0.5)) for c in LABELS]
t = pd.DataFrame(tab, index=LABELS); t.loc["MEAN"] = t.mean()
print(t.round(3).to_string())

out = soft.copy(); out.insert(0, "is_gold", is_gold.astype(int)); out["fold"] = -1
weak = ~is_gold.values
groups = TRAIN.Report.map(lambda x: re.sub(r"\W+", "", str(x).lower())).values[weak]
order = np.random.RandomState(0).permutation(weak.sum())
folds = np.zeros(weak.sum(), int)
for k, (_, va) in enumerate(GroupKFold(5).split(order, groups=groups[order])):
    folds[order[va]] = k
out.loc[weak, "fold"] = folds
for c in LABELS:
    out[f"gold_{c}"] = gold[c]
out["consensus_mean"] = cons.mean(1).values
out["teacher_mean"] = teacher.mean(1).values
out.to_csv("/kaggle/working/labels_student_v0.csv")
say("labels saved; fold sizes", out.fold.value_counts().sort_index().to_dict())
say("mean soft:", out[LABELS].mean().round(2).to_dict())


In [ ]:

import os, json, subprocess, time, glob, shutil
assert os.environ.get("KAGGLE_USERNAME"), "env kaggle tidak ada"
DS_ID = f"{os.environ['KAGGLE_USERNAME']}/rsna-knee-student-prep"
meta = {
    "id": DS_ID, "title": "RSNA Knee student prep cache256",
    "licenses": [{"name": "Other"}], "keywords": ["rsna", "knee", "cache"], "resources": []
}
os.makedirs("/tmp/ds_push", exist_ok=True)
with open("/tmp/ds_push/dataset-metadata.json", "w") as f:
    json.dump(meta, f)
shutil.copy("/kaggle/working/cache256_train_meta.csv", "/tmp/ds_push/")
shutil.copy("/kaggle/working/labels_student_v0.csv", "/tmp/ds_push/")
# taruh cache di subfolder (dataset CLI push folder)
if os.path.exists("/tmp/cache256_train"):
    os.system("ln -sfn /tmp/cache256_train /tmp/ds_push/cache256_train")
say("pushing dataset (bisa lama)...")
r = subprocess.run(["kaggle", "datasets", "version", "-p", "/tmp/ds_push", "-m", "cache256 + labels student v0"],
                   capture_output=True, text=True)
say("push rc", r.returncode); say(r.stdout[-500:]); say(r.stderr[-500:])
say("DONE prep")
